# NexteRaS — Personal CPU Learning Lab: Fine-Tuning `sentence-transformers/all-MiniLM-L6-v2`

**Hardware Target:** Windows Laptop · Intel Core i5-1334U (CPU) · Intel Iris Xe (No CUDA) · 15.6 GB RAM

**Objective:** Learn how domain adaptation with `MultipleNegativesRankingLoss` affects semantic retrieval between synthetic Indian student profiles and internship/opportunity descriptions — and compare **baseline pretrained performance** vs **post-training performance** honestly on held-out examples.

## 1. Verify Environment & CPU-Only Runtime
Confirm Python 3.11 inside `ml/.venv` and verify PyTorch is using CPU (`torch.cuda.is_available() == False`).

In [ ]:
import sys
from pathlib import Path

# Resolve repository root regardless of whether Jupyter launches from repo root or ml/notebooks
cwd = Path.cwd().resolve()
REPO_ROOT = next(p for p in [cwd, *cwd.parents] if (p / 'ml' / 'model_configs').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml.src.common.config import configure_cpu_runtime, load_yaml_config, resolve_repo_path, seed_everything

cfg = load_yaml_config('ml/model_configs/personal_minilm_cpu.yaml')
seed_everything(cfg['seed'])
runtime = configure_cpu_runtime(cfg['hardware']['cpu_threads'])
print('Python:', sys.version.split()[0])
print('Runtime:', runtime)

## 2. Inspect & Prepare Synthetic Dataset (Zero Data Leakage)
We generate disjoint `train`, `val`, and held-out `test` splits from `ml/datasets/synthetic/matching_seed_examples.json`. No student ID or positive opportunity ID overlaps across splits.

In [ ]:
import json
import pandas as pd
from ml.src.data.matching_dataset import write_processed_splits

seed_path = resolve_repo_path(cfg['data']['seed_file'])
processed_dir = resolve_repo_path(cfg['data']['processed_dir'])
counts = write_processed_splits(seed_path, processed_dir)
print('Prepared counts (leakage-checked):', counts)

train_rows = [json.loads(l) for l in resolve_repo_path(cfg['data']['train_file']).read_text(encoding='utf-8').splitlines() if l]
val_rows = [json.loads(l) for l in resolve_repo_path(cfg['data']['val_file']).read_text(encoding='utf-8').splitlines() if l]
test_queries = [json.loads(l) for l in resolve_repo_path(cfg['data']['test_queries_file']).read_text(encoding='utf-8').splitlines() if l]
test_corpus = [json.loads(l) for l in resolve_repo_path(cfg['data']['test_corpus_file']).read_text(encoding='utf-8').splitlines() if l]

pd.DataFrame(train_rows)[['pair_id', 'student_id', 'opportunity_id', 'is_synthetic']].head()

## 3. Baseline Evaluation BEFORE Fine-Tuning (~10s after ~90MB Hugging Face download)
We measure `Recall@1/3/5`, `MRR@10`, `nDCG@10`, and positive-vs-hard-negative cosine separation **before** updating any weights.

In [ ]:
from sentence_transformers import SentenceTransformer
from ml.src.evaluation.retrieval_metrics import evaluate_sentence_transformer

model = SentenceTransformer(cfg['model']['base_model_name'], device='cpu')
model.max_seq_length = cfg['model']['max_sequence_length']

baseline_metrics = evaluate_sentence_transformer(model, test_queries, test_corpus)
pd.Series(baseline_metrics, name='baseline')

## 4. CPU Fine-Tuning with `MultipleNegativesRankingLoss` (~1.5–4 min on i5-1334U CPU)
Batch size is kept at `8` for comfortable memory (~1.5 GB RSS) on this 15.6 GB Windows laptop.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

train_ds = Dataset.from_list([{'anchor': r['anchor'], 'positive': r['positive']} for r in train_rows])
val_ds = Dataset.from_list([{'anchor': r['anchor'], 'positive': r['positive']} for r in val_rows])
output_dir = resolve_repo_path(cfg['output']['model_dir'])

args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / '_trainer_tmp'),
    num_train_epochs=cfg['training']['epochs'],
    per_device_train_batch_size=cfg['training']['batch_size'],
    per_device_eval_batch_size=cfg['training']['batch_size'],
    learning_rate=cfg['training']['learning_rate'],
    warmup_ratio=cfg['training']['warmup_ratio'],
    weight_decay=cfg['training']['weight_decay'],
    fp16=False,
    bf16=False,
    use_cpu=True,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='no',
    report_to='none',
    seed=cfg['seed'],
    data_seed=cfg['seed'],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=MultipleNegativesRankingLoss(model=model),
)

trainer.train()
model.save_pretrained(str(output_dir))
print('Saved fine-tuned checkpoint to:', output_dir)

## 5. Held-Out Comparison & Visualization (Before vs. After Fine-Tuning)
Verify directly on `test_queries` & `test_corpus` whether domain adaptation improved or degraded each retrieval metric.

In [ ]:
import matplotlib.pyplot as plt

finetuned_metrics = evaluate_sentence_transformer(model, test_queries, test_corpus)
metric_keys = ['recall_at_1', 'recall_at_3', 'recall_at_5', 'mrr_at_10', 'ndcg_at_10', 'mean_pos_neg_margin']
comp_df = pd.DataFrame({
    'baseline': [baseline_metrics[k] for k in metric_keys],
    'finetuned': [finetuned_metrics[k] for k in metric_keys],
}, index=metric_keys)
comp_df['delta'] = comp_df['finetuned'] - comp_df['baseline']
display(comp_df)

comp_df[['baseline', 'finetuned']].plot(kind='bar', figsize=(9, 4), title='NexteRaS all-MiniLM-L6-v2: Baseline vs Fine-Tuned (Synthetic Held-Out Split)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 6. Inspect Concrete Query Predictions & Limitations
- **Limitations:** Our starter dataset is synthetic and small (`20` train pairs, `8` test queries). Small positive gains show the training loop adapted wording weights without breaking generalization, whereas overfitting shows up if `val` loss rises or held-out margin shrinks.
- **Deterministic Boundary:** Even with a 0.99 cosine score, hard eligibility constraints (CGPA, graduation batch, degree) must still be enforced deterministically in Python!

In [ ]:
from ml.src.matching.semantic_matcher import SemanticMatcher

matcher = SemanticMatcher(output_dir, device='cpu')
sample_q = test_queries[0]
print('Query student:', sample_q['query_id'], '| Expected target:', sample_q['positive_doc_id'])
for rank, item in enumerate(matcher.rank_opportunities(sample_q['query_text'], test_corpus, top_k=3), start=1):
    print(f'  #{rank}: {item.doc_id} | {item.title} | cosine={item.score}')